# Model 0 — Oxygen-mediated motility and aggregation

**C. elegans-inspired agent rules · a minimal mechanism experiment**

The central question is: **can agents aggregate because they change oxygen, and oxygen changes their speed?**

Every agent follows the same rule. It has a position and heading, consumes oxygen locally, and moves at a speed set by the oxygen at its position. Its turning is random and unbiased. Agents can overlap freely.

We will first understand accumulation in a prescribed oxygen landscape, then let the agents create their own landscape. Finally, we will remove individual couplings and compare the outcomes.

This notebook follows the teaching structure of SwarmRules: explanations and experiments here, technical implementation in `celegans/model0/`. The original SwarmRules files are unchanged.

## 1. Scientific motivation and scope

Cates and Tailleur describe how self-propelled particles can accumulate where they move slowly. If accumulation also causes slowdown, the resulting feedback can destabilize a uniform population [1].

Demir et al. connect C. elegans patterns to oxygen-dependent motility and bacteria-associated oxygen depletion. Their measured speed response is not globally increasing: movement rises again at very low oxygen, and the response differs between strains and feeding conditions [2, Figure 2].

**Our choice:** isolate one increasing-speed branch with a smooth, positive toy law. Model 0 does not reproduce the low-oxygen escape response or use measured parameter values. Consumption is a phenomenological agent-associated sink; it is not an estimate of worm respiration alone. No explicit bacteria field is included.

| Representation | Included in Model 0 | Left for later models |
|---|---|---|
| Animal | Point position, heading, oxygen-dependent speed | Flexible centerline, body length, reversals |
| Interactions | Shared oxygen field; free overlap | Contact forces, contact slowdown, alignment |
| Oxygen response | Change of speed; constant angular diffusivity | Explicit biased turning or sensory memory |
| Environment | Oxygen diffusion, replenishment, consumption | Bacteria, feeding, food depletion |

The output may show **aggregation**. Establishing phase separation requires additional evidence, and coherent swarming requires more than dense patches.

## 2. Start here

Install `requirements.txt` in the Python environment used by this notebook. Run cells in order. Each interactive panel starts paused; use **Start**, **Pause**, **Advance frame**, and **Reset**.

- Physical controls apply immediately. Reset after changing a parameter for a clean comparison.
- Seed and the stopping turn are run settings. The seed is applied on Reset.
- Steps / draw changes viewing cadence only. It does not change numerical `dt`.
- A new demo pauses a previously running demo. Pause before long headless experiments.
- Static reference figures below remain visible without running a kernel.

All lengths and times are **arbitrary model units**. Oxygen is divided by a fixed reference concentration; a value of 0.5 is not a biological oxygen percentage.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
from IPython.display import display
import celegans as ce

print("Model 0 version:", ce.__version__)
ce.info()
p = ce.Parameters()

## 3. Define one agent

Agent $i$ has position $\mathbf r_i=(x_i,y_i)$ and heading $\theta_i$, with $\mathbf p_i=(\cos\theta_i,\sin\theta_i)$:

$$
d\mathbf r_i=v(O_i)\mathbf p_i\,dt,
\qquad d\theta_i=\sqrt{2D_r}\,dW_i.
$$

Each $W_i$ is an independent Wiener process. Over a numerical interval $h$, angular noise has standard deviation $\sqrt{2D_r h}$. Thus $D_r$ is a rate, not a noise amplitude per frame. Small $D_r$ gives long directional persistence; for $D_r>0$ the heading correlation time is $1/D_r$.

$O_i$ is sampled by bilinear interpolation of the local oxygen field. There is **no gradient-dependent turning term**. The agents have no target, map, or alignment instruction. Opposite edges of the domain are connected, so both agents and oxygen obey periodic boundaries.

### Oxygen-dependent speed

We choose the teaching law

$$
v(O)=v_{\min}+(v_{\max}-v_{\min})
\frac{1}{1+\exp[-(O-O_c)/w]}.
$$

$O_c$ sets the transition midpoint and $w$ its width. Smaller $w$ gives a sharper response. The positive floor $v_{\min}$ avoids making low-oxygen regions permanently absorbing. This law is an assumption we can later replace with a measured curve.

Setting `speed_response=False` makes $v(O)=v_{\max}$ everywhere; oxygen then cannot affect the agents.

In [ ]:
oxygen_values = np.linspace(0, 1, 300)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(oxygen_values, ce.speed(oxygen_values, p), color="#087e8b", lw=2.5)
ax.axvline(p.oxygen_midpoint, color="0.5", ls=":", label="Response midpoint")
ax.set(xlabel="Normalized oxygen O", ylabel="Speed [model length / time]",
       title="The assumed speed law — not an experimental fit")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.show()

## 4. Experiment 1 — a prescribed oxygen landscape

**Question:** Can agents accumulate without being instructed to turn toward a location?

Here the field is held fixed:

$$
O(x)=0.55+0.25\cos(2\pi x/L_x).
$$

Agents start uniformly in position and heading. Only their speed changes as they cross the landscape. The oxygen equation is switched off, so this experiment has no population-to-oxygen feedback.

For this test we use a smaller $32\times24$ domain, 900 agents, and $v_{\min}=0.3$ to make exploration of the low-speed region faster. Other parameters retain their defaults. These choices are explicit experiment settings, not biological measurements.

For a fixed positive speed landscape, constant $D_r>0$, no translational diffusion, and periodic boundaries, an isotropic stationary solution is

$$\rho(x)\propto\frac{1}{v(O(x))}.$$

Think of residence time: a slow region is occupied longer. This is accumulation in an externally imposed landscape, not spontaneous pattern formation.

In [ ]:
landscape_params = replace(p, width=32, height=24, nx=32, ny=24,
                           n_agents=900, v_min=0.3)
landscape_demo = ce.Demo("landscape", landscape_params, seed=1, turns=12000)
landscape_demo.show()

### Predict, observe, compare

1. Predict where density will be greatest before pressing Start.
2. Disable **Speed depends on oxygen**, Reset, and compare.
3. Restore the response and vary angular diffusion. Does the stationary prediction change, or mainly the relaxation time?
4. Change the response width. How does the density contrast respond?

The next cell measures a finite run and compares its profile with the stationary prediction. Sampling noise and incomplete relaxation are expected; the dashed curve is not a fit to the output. It uses the programmed fixed field and speed law.

In [ ]:
landscape_demo.pause()
landscape_result = ce.run("landscape", landscape_params, seed=1, turns=12000)
x, observed = ce.x_profile(landscape_result, bins=16)
# Integrate the stationary prediction over the same bins.
fine_x = (np.arange(1600) + 0.5) * landscape_params.width / 1600
prediction = 1 / ce.speed(0.55 + 0.25*np.cos(2*np.pi*fine_x/landscape_params.width),
                         landscape_params)
prediction = prediction.reshape(16, -1).mean(axis=1)
prediction /= prediction.mean()
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(x, observed, "o-", label="Finite simulation", color="#087e8b")
ax.plot(x, prediction, "--", label="Stationary target: density proportional to 1 / speed",
        color="#d66b32")
ax.set(xlabel="x [model length]", ylabel="Density / mean", ylim=(0, None))
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.2)
plt.show()

## 5. Add a dynamic oxygen field

Now agents change their environment:

$$
\partial_t O=D_O\nabla^2O+k(O_{\mathrm{amb}}-O)
-q\rho_K(\mathbf x,t)\frac{O}{K_O+O}.
$$

| Term | Meaning |
|---|---|
| $D_O\nabla^2O$ | Lateral oxygen diffusion |
| $k(O_{\mathrm{amb}}-O)$ | Exchange with an external reservoir |
| $q\rho_K O/(K_O+O)$ | Agent-associated consumption, reduced when oxygen is scarce |

$\rho_K$ is agent density spread over a normalized Gaussian footprint of standard deviation $\ell_K$. Its integral equals the number of agents. This spreads a point sink over a finite length; it does not give agents an excluded-volume body. $K_O$ is the oxygen half-saturation concentration and is unrelated to the subscript on $\rho_K$.

The consumption form is our assumption. It prevents a finite sink from continuing to remove oxygen at zero concentration. The speed floor and consumption saturation regularize different parts of the model.

### Close the feedback loop

A small excess in density can reduce oxygen, lower speed, and increase residence time. Diffusion, replenishment, persistence, and the response curve determine whether this local excess grows or dissolves. Strong consumption can also slow almost the whole population; **low average speed alone is not evidence of aggregation**.

### An important connection to the earlier discussion

Even with unbiased turning, coarse-graining these variable-speed agents gives, at long times and leading gradient order,

$$
\mathbf J\simeq-\frac{v}{2D_r}\nabla(v\rho)
=-\frac{v^2}{2D_r}\nabla\rho
-\frac{vv'}{2D_r}\rho\nabla O.
$$

Thus an oxygen-gradient contribution to the population flux can arise from **speed-dependent residence time alone**. Adding an explicit directional steering rule would introduce an additional mechanism; it is not needed merely to obtain this term. This relation assumes sufficiently slow variations relative to heading relaxation and does not replace the agent simulation.

## 6. Parameters and the meaning of one turn

These are the defaults used in the feedback experiment. Sliders expose a subset; all values are available through `ce.Parameters`.

| Quantity | Python name | Default | Role / units |
|---|---|---:|---|
| $L_x,L_y$ | `width`, `height` | 64, 48 | Domain lengths |
| $N$ | `n_agents` | 1800 | Number of agents |
| $v_{\min},v_{\max}$ | `v_min`, `v_max` | 0.06, 2.0 | Length / time |
| $O_c$ | `oxygen_midpoint` | 0.55 | Speed transition concentration |
| $w$ | `response_width` | 0.045 | Speed transition width |
| $D_r$ | `rotational_diffusion` | 0.25 | Inverse time |
| $D_O$ | `oxygen_diffusion` | 1.0 | Length squared / time |
| $k$ | `replenishment` | 0.06 | Inverse time |
| $O_{\mathrm{amb}}$ | `ambient` | 1.0 | Normalized reservoir concentration |
| $q$ | `consumption` | 0.06 | Concentration × area / (agent × time) |
| $K_O$ | `oxygen_half_saturation` | 0.10 | Consumption half-saturation |
| $\ell_K$ | `kernel_width` | 1.2 | Consumption footprint length |

**Numerical settings:** the grid has `nx=64`, `ny=48` cells; `dt=0.1` model time per outer turn. Grid spacing is domain length divided by cell count. Consumption density is divided by cell area, so grid refinement preserves total agent count and the meaning of $q$.

For each internal interval $h$, the engine:

1. constructs consumption density and samples oxygen at current positions;
2. adds independent angular increments of variance $2D_rh$;
3. moves agents with the sampled speeds and updated headings, wrapping positions;
4. advances oxygen using the beginning-of-interval density: diffuse, replenish, consume.

The diffusion step uses a periodic five-point Laplacian. Replenishment is exact over $h$; consumption is solved implicitly and remains nonnegative. Internal substeps obey the diffusion stability bound and limit displacement and reaction changes. This first-order split scheme is stable under its step restrictions, but quantitative claims still need time-step and grid-convergence checks.

**Run settings:** `seed`, `turns`, and diagnostic sampling interval `record_every`. **Display setting:** `steps / draw`. Changing the latter does not change trajectories.

## 7. Experiment 2 — let the population create the landscape

**Question:** Do initially small density fluctuations grow under the full coupling?

Agents start uniformly at random. Oxygen starts at the homogeneous reaction balance for the mean density $\bar\rho=N/(L_xL_y)$:

$$k(O_{\mathrm{amb}}-O_0)=q\bar\rho\frac{O_0}{K_O+O_0}.$$

This avoids spending the beginning of every run depleting an initially oxygen-rich domain. It does not insert an oxygen pattern or an agent cluster. The random initial positions provide finite-population fluctuations.

The four panels show oxygen plus agents, smoothed density, density heterogeneity, and mean oxygen/speed. White dashes indicate a deterministic subset of headings; they are not flexible worm bodies. Map scales stay fixed for comparison; density colors saturate above four times the mean.

In [ ]:
print(f"Homogeneous initial oxygen: {ce.uniform_oxygen(p):.3f}")
feedback_demo = ce.Demo("feedback", p, seed=1, turns=3000)
feedback_demo.show()

### Reference run and exploration

This is an actual simulation with the listed default parameters, seed 1, and 3000 turns ($t=300$). It is a qualitative reference, not a morphology target.

![Reference run of Model 0](assets/feedback_reference.png)

Before tuning several controls at once, make a prediction:

- What should happen if consumption is zero?
- What should faster replenishment or oxygen diffusion do to local oxygen contrasts?
- Does making the speed transition sharper strengthen the feedback near its midpoint?
- Can you obtain a population that is almost uniformly slow rather than clustered?
- Does reducing angular diffusion always increase aggregation, or does long persistence allow agents to escape small regions?

Change one control, Reset, and compare at the same final time. Keep the parameter set, seed, and time when recording an observation. Slider limits are exploratory choices, not fitted biological ranges or special success bounds.

## 8. Experiment 3 — remove one coupling at a time

**Question:** Is the pattern caused by oxygen-dependent motility, rather than by an unintended attractive force or an initially dense patch?

Compare three runs with identical initial agent positions and headings (same seed and geometry):

| Run | Consumption | Speed response | Interpretation |
|---|---|---|---|
| Full feedback | On | On | Complete mechanism |
| Constant speed | On | Off | Agents reshape oxygen, but cannot respond to it |
| No consumption | Off | On | Oxygen stays spatially uniform |

Each dynamic run starts at its own homogeneous oxygen balance; the no-consumption control therefore starts at ambient oxygen. The constant-speed control shares the full-feedback initial oxygen field. Controls may have different mean speeds, which is part of the ablation, not evidence of a density pattern.

We measure

$$H(t)=\left\langle\left(\frac{\rho_{\mathrm{obs}}}{\bar\rho}-1\right)^2\right\rangle_{\mathbf x},$$

where $\rho_{\mathrm{obs}}$ uses a **fixed observation length of 2 model units**, independent of the consumption footprint. Uniform random agents have a nonzero finite-number baseline. Compare against controls with the same population, domain, and observation length rather than treating $H>0$ as an instability test.

In [ ]:
feedback_demo.pause()
controls = {
    "Full feedback": p,
    "Constant speed": replace(p, speed_response=False),
    "No consumption": replace(p, consumption=0.0),
}
results = {label: ce.run("feedback", params, seed=1, turns=3000)
           for label, params in controls.items()}
display(ce.compare(results))
for label, result in results.items():
    print(f"{label:16s} H(initial)={result.history['heterogeneity'][0]:.3f} "
          f"H(final)={result.history['heterogeneity'][-1]:.3f} "
          f"polar order={result.history['polar_order'][-1]:.3f}")

![Controlled comparison reference](assets/control_comparison.png)

**Task:** Explain the difference between a density cluster and a motile swarm. The polar order $P=|N^{-1}\sum_i e^{i\theta_i}|$ measures global heading alignment; random headings give a finite-size value of order $N^{-1/2}$. This model has no alignment torque. A large $H$ with small $P$ is compatible with aggregation without global alignment. A translating density pattern would require a separate measurement of pattern motion; $P$ alone does not settle that question.

### Reproducibility across seeds

The next cell repeats the full coupling and the constant-speed control. It is a small robustness check, not a phase diagram. Compare the growth relative to each run's initial baseline.

In [ ]:
seed_rows = []
for seed in (1, 2, 3):
    for label in ("Full feedback", "Constant speed"):
        result = results[label] if seed == 1 else ce.run(
            "feedback", controls[label], seed=seed, turns=3000)
        h = result.history["heterogeneity"]
        seed_rows.append((seed, label, h[0], h[-1]))
print("seed  condition          H(0)     H(300)")
for seed, label, h0, hf in seed_rows:
    print(f"{seed:4d}  {label:16s}  {h0:7.3f}  {hf:7.3f}")

## 9. Experiment 4 — does an imposed density excess persist?

**Question:** Can feedback maintain an existing density perturbation, and does it dissolve when the speed response is removed?

The `seeded` scenario places 20% of agents in a Gaussian patch at the domain center; the rest are uniform. Oxygen still starts spatially uniform. This is a relaxation experiment. Do not describe its initially dense patch as spontaneous aggregation.

Run the same scenario with `speed_response=False` and compare how $H$ evolves. Keep the same seed and final time. You may also try changing only population size at fixed domain size, then ask why a density sweep changes both oxygen balance and crowding statistics.

In [ ]:
seeded_demo = ce.Demo("seeded", p, seed=1, turns=3000)
seeded_demo.show()

# Example control, when you are ready:
# seeded_demo.pause()
# seeded_control = ce.run("seeded", replace(p, speed_response=False), seed=1, turns=3000)
# display(ce.plot(seeded_control))

## 10. Inspect and control the simulation programmatically

`Simulation` is the same engine used by `Demo` and `run`. `state` contains independent read-only snapshots of positions `(N, 2)`, headings `(N,)`, and oxygen `(ny, nx)`. Unlike a live view, an earlier snapshot does not change when the simulation advances. `result()` adds diagnostic histories and parameter metadata.

Use this interface to compute an order parameter, stop on a condition, or change a parameter during a run. A mid-run change is a perturbation experiment; use Reset for an independent comparison. Structural changes to the population, grid, domain or `dt` require a new simulation.

In [ ]:
seeded_demo.pause()
sim = ce.Simulation("feedback", p, seed=4)
sim.step(100)
state = sim.state
print("positions:", state.positions.shape, "oxygen:", state.oxygen.shape)
print("time:", state.time, "mean oxygen:", state.oxygen.mean())

sim.set_params(consumption=0.03)  # a recorded mid-run intervention
sim.step(100)
result = sim.result()
print("parameter configurations recorded:", len(result.parameter_log))
# sim.reset(seed=5) restarts with the current parameters.
# state.positions.copy() makes a writable independent array for analysis.

## 11. Numerical and scientific checks before extending the model

The release includes automated tests of density conservation, periodic boundaries,
positive oxygen, analytic diffusion/replenishment limits, fixed fields, repeatable
reset, and independence from how turns are grouped for rendering. See `VALIDATION.md`.

For a quantitative study, check ensemble statistics at the **same physical final time** while reducing `dt` and refining the grid. Preserve physical dimensions, footprint length, and population density. Grid refinement changes `nx, ny`, not `width, height`. Different discretizations can consume random numbers differently, so compare distributions across seeds, not exact trajectories.

```python
# Optional convergence experiment; this is slower than the main exercises.
variants = {
    "base": p,
    "half dt": replace(p, dt=p.dt / 2),
    "finer grid": replace(p, nx=2*p.nx, ny=2*p.ny, dt=p.dt / 2),
}
for name, parameters in variants.items():
    values = [ce.run("feedback", parameters, seed=s,
                     turns=round(300 / parameters.dt)).history["heterogeneity"][-1]
              for s in (1, 2, 3)]
    print(name, np.mean(values), np.std(values, ddof=1))
```

**Interpretation limits:** a cluster is not by itself proof of bulk phase separation. Finite size, observation length, oxygen diffusion length, and the consumption footprint all affect morphology. A sharper scientific claim would need time-dependent coarsening measures, density distributions, system-size comparisons, and evidence of stationary behavior or phase coexistence. The point agents can build arbitrarily high local density; a finite-speed floor does not impose a packing limit.

The next model should add a mechanism because an observation requires it. Candidates include a measured nonmonotone speed curve, explicit directional oxygen sensing, worm geometry, contact rules, or bacteria dynamics. This notebook establishes the baseline against which those additions can be tested.

## References and source map

1. Cates, M. E. & Tailleur, J. *Motility-Induced Phase Separation.* Annual Review of Condensed Matter Physics **6**, 219–244 (2015; preprint 2014). [arXiv:1406.3533](https://arxiv.org/abs/1406.3533), [DOI](https://doi.org/10.1146/annurev-conmatphys-031214-014710).
2. Demir, E., Yaman, Y. I., Basaran, M. & Kocabas, A. *Dynamics of pattern formation and emergence of swarming in Caenorhabditis elegans.* eLife **9**, e52781 (2020). [Article](https://elifesciences.org/articles/52781), [PDF](https://elifesciences.org/articles/52781.pdf). Figure 2 and the associated discussion motivate the oxygen–motility connection; our parameterization is not a fit to that figure.

The implementation separates mechanisms (`model.py`), experiment setup (`scenarios.py`), measurements (`analysis.py`), drawing (`visualization.py`) and interactive controls (`demo.py`). Ordinary experiments need only `import celegans as ce`.

No published figures or measured datasets are included. All displayed simulation references were generated with this package.